# Coordinator NB03 — `predict()` Implementation

Implements the coordinator architecture from NB02 empirical tests, updated to produce
a directional call on **all 4 pairs** using a two-tier confidence system.

**Architecture:**
- **VOL track (fwd_vol_3d):** StockTwits (USDJPY active days, ~40%) → else geo (all pairs)
- **DIRECTIONAL track:**
  - `"high"` — USDJPY + tech confidence ≥ p75 (accuracy=58.6%, binomial p=0.014); horizon=1d
  - `"medium"` — GBPUSD macro direction (IC_5d=+0.115, p<0.001); horizon=5d
  - `"low"` — EURUSD / USDCHF / USDJPY-fallback macro direction (IC_5d=0.03–0.06); horizon=5d
  - `None` only when macro_direction is literally missing (data gap)
- **REGIME overlay:** Google Trends `macro_attention_zscore` (Tier B, in-sample only)

**Known constraints:**
- Zone collapse: `geo_bilateral_risk` identical across all 4 pairs → global stress indicator, not pair-specific
- USDJPY tech model always outputs direction=1 → constant signal, validated by accuracy (not IC)
- Macro direction validated at 5d horizon; tech_usdjpy validated at 1d — horizons are not comparable

In [1]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

SIGNALS = ROOT / "data/processed/coordinator/signals_aligned.parquet"
df = pd.read_parquet(SIGNALS)
df["date"] = pd.to_datetime(df["date"])

print(f"{len(df):,} rows | {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Pairs: {sorted(df['pair'].unique())}")
print(f"Columns: {list(df.columns)}")

4,172 rows | 2022-01-03 → 2025-12-31
Pairs: ['EURUSD', 'GBPUSD', 'USDCHF', 'USDJPY']
Columns: ['date', 'pair', 'tech_direction', 'tech_confidence', 'tech_vol_regime', 'geo_bilateral_risk', 'geo_risk_regime', 'macro_direction', 'macro_confidence', 'macro_carry_score', 'macro_regime_score', 'macro_fundamental_score', 'macro_surprise_score', 'macro_bias_score', 'macro_dominant_driver', 'usdjpy_stocktwits_vol_signal', 'gdelt_tone_zscore', 'gdelt_attention_zscore', 'macro_attention_zscore', 'composite_stress_flag', 'fwd_ret_1d', 'fwd_ret_5d', 'fwd_vol_3d', 'fwd_vol_5d', 'fwd_vol_10d']


## 1. CoordinatorSignal Schema

In [2]:
@dataclass
class CoordinatorSignal:
    pair: str
    date: pd.Timestamp
    # VOL track
    vol_signal: float | None      # raw vol signal (see vol_source for sign convention)
    vol_source: str               # "stocktwits" | "geo" | "none"
    # DIRECTIONAL track
    direction: int | None         # 1=up (base appreciates), 0=down, None=no data
    direction_source: str         # "tech_usdjpy" | "macro_{pair}" | "flat"
    direction_horizon: str        # "1d" | "5d" | "none"
    direction_ic: float | None    # Spearman IC at chosen horizon; None if N/A (constant signal)
    confidence_tier: str          # "high" | "medium" | "low" | "none"
    flat_reason: str | None       # populated only when direction is None
    # REGIME overlay
    regime: str                   # "high_attention" | "normal" | "unknown"

    def __repr__(self) -> str:
        vol = f"{self.vol_signal:.4f}" if self.vol_signal is not None else "None"
        dir_str = {1: "UP", 0: "DOWN", None: "no-data"}.get(self.direction, "?")
        ic_str = f"{self.direction_ic:+.3f}" if self.direction_ic is not None else "n/a"
        return (
            f"CoordinatorSignal({self.pair} {self.date.date()} | "
            f"vol={vol}[{self.vol_source}] | "
            f"{dir_str}[{self.direction_source},tier={self.confidence_tier},"
            f"h={self.direction_horizon},ic={ic_str}] | regime={self.regime})"
        )

print("CoordinatorSignal defined")

CoordinatorSignal defined


## 2. Coordinator `predict()` — Fusion Rules

### VOL fusion
```
if pair == USDJPY and StockTwits active (40% of days):
    vol_signal = usdjpy_stocktwits_vol_signal   ← IC=−0.30 vs fwd_vol_3d (inversely signed)
    vol_source = "stocktwits"
else:
    vol_signal = geo_bilateral_risk             ← IC=+0.13 vs fwd_vol_3d
    vol_source = "geo"
```

### DIRECTIONAL fusion — two-tier + macro fallback (all pairs always get a call)
```
USDJPY + tech_confidence ≥ p75:
    direction = tech_direction (always 1 — model constant)
    confidence_tier = "high"    ← accuracy=58.6%, binomial p=0.014
    direction_horizon = "1d"
    direction_ic = None          ← constant signal; accuracy is the right metric

All other cases (incl. USDJPY when gate doesn't fire):
    direction = 1 if macro_direction=="up" else 0
    direction_source = "macro_{pair}"
    direction_horizon = "5d"
    confidence_tier:
        "medium" → GBPUSD (IC_5d=+0.115, p<0.001)
        "low"    → EURUSD (IC_5d=+0.060), USDCHF (IC_5d=+0.033), USDJPY-fallback (IC_5d=+0.058)

direction = None only when macro_direction itself is missing (rare data gap).
```

### REGIME overlay
```
macro_attention_zscore > 1.0 → "high_attention"  (IC=+0.087 vs fwd_vol_3d, Tier B in-sample)
```

In [3]:
# Compute USDJPY tech confidence p75 from training window (2022-01-01 to 2023-12-31)
TRAIN_END = pd.Timestamp("2023-12-31")

usdjpy_train = df[(df["pair"] == "USDJPY") & (df["date"] <= TRAIN_END)]
USDJPY_CONF_P75 = float(usdjpy_train["tech_confidence"].quantile(0.75))
print(f"USDJPY tech_confidence p75 (train window): {USDJPY_CONF_P75:.6f}")
print(f"  Training rows: {len(usdjpy_train)}")
print(f"  Days above gate in training: {(usdjpy_train['tech_confidence'] >= USDJPY_CONF_P75).sum()} ({(usdjpy_train['tech_confidence'] >= USDJPY_CONF_P75).mean():.1%})")

USDJPY tech_confidence p75 (train window): 0.088708
  Training rows: 520
  Days above gate in training: 130 (25.0%)


In [4]:
# ── Per-pair macro directional config: (confidence_tier, horizon, direction_ic) ──
# ICs from pre-implementation validation (macro_dir_enc vs fwd_ret at horizon):
#   GBPUSD  5d: IC=+0.115, p<0.001  → "medium"
#   EURUSD  5d: IC=+0.060, p=0.055  → "low"
#   USDCHF  5d: IC=+0.033, p=0.296  → "low"
#   USDJPY  5d: IC=+0.058, p=0.063  → "low" (macro fallback when tech gate doesn't fire)
# USDJPY tech gate: always outputs direction=1 (constant) → validated by accuracy, not IC
PAIR_MACRO_CONFIG: dict[str, tuple[str, str, float]] = {
    "GBPUSD": ("medium", "5d", 0.115),
    "EURUSD": ("low",    "5d", 0.060),
    "USDCHF": ("low",    "5d", 0.033),
    "USDJPY": ("low",    "5d", 0.058),
}

print("Per-pair macro config (tier, horizon, IC_5d):")
for pair, (tier, horizon, ic_val) in PAIR_MACRO_CONFIG.items():
    print(f"  {pair}: tier={tier!r}, horizon={horizon}, IC={ic_val:+.3f}")
print(f"\nUSYDJPY tech gate: constant signal → validated by accuracy=0.586, direction_ic=None")

Per-pair macro config (tier, horizon, IC_5d):
  GBPUSD: tier='medium', horizon=5d, IC=+0.115
  EURUSD: tier='low', horizon=5d, IC=+0.060
  USDCHF: tier='low', horizon=5d, IC=+0.033
  USDJPY: tier='low', horizon=5d, IC=+0.058

USYDJPY tech gate: constant signal → validated by accuracy=0.586, direction_ic=None


In [5]:
REGIME_ATTENTION_THRESHOLD = 1.0  # macro_attention_zscore > 1.0 → high_attention


def coordinator_predict(row: pd.Series, usdjpy_conf_p75: float = USDJPY_CONF_P75) -> CoordinatorSignal:
    """Map one signal table row to a CoordinatorSignal."""
    pair = str(row["pair"])
    date = pd.Timestamp(row["date"])

    # ── VOL track ────────────────────────────────────────────────────────────
    stocktwits_val = row.get("usdjpy_stocktwits_vol_signal")
    if pair == "USDJPY" and pd.notna(stocktwits_val):
        vol_signal = float(stocktwits_val)
        vol_source = "stocktwits"
    else:
        geo_val = row.get("geo_bilateral_risk")
        vol_signal = float(geo_val) if pd.notna(geo_val) else None
        vol_source = "geo" if vol_signal is not None else "none"

    # ── DIRECTIONAL track ────────────────────────────────────────────────────
    flat_reason: str | None = None

    if pair == "USDJPY":
        conf = row.get("tech_confidence")
        if pd.notna(conf) and float(conf) >= usdjpy_conf_p75:
            # High-confidence tech gate — accuracy=58.6%, binomial p=0.014
            direction: int | None = int(row["tech_direction"])
            direction_source = "tech_usdjpy"
            direction_horizon = "1d"
            direction_ic: float | None = None  # constant signal; IC undefined
            confidence_tier = "high"
        else:
            # Macro fallback for USDJPY non-gated days
            macro_dir = row.get("macro_direction")
            if pd.notna(macro_dir):
                direction = 1 if macro_dir == "up" else 0
                tier, horizon, ic_val = PAIR_MACRO_CONFIG["USDJPY"]
                direction_source = "macro_usdjpy"
                direction_horizon = horizon
                direction_ic = ic_val
                confidence_tier = tier
            else:
                direction = None
                direction_source = "flat"
                direction_horizon = "none"
                direction_ic = None
                confidence_tier = "none"
                flat_reason = "macro_direction unavailable"
    else:
        macro_dir = row.get("macro_direction")
        if pd.notna(macro_dir):
            direction = 1 if macro_dir == "up" else 0
            tier, horizon, ic_val = PAIR_MACRO_CONFIG[pair]
            direction_source = f"macro_{pair.lower()}"
            direction_horizon = horizon
            direction_ic = ic_val
            confidence_tier = tier
        else:
            direction = None
            direction_source = "flat"
            direction_horizon = "none"
            direction_ic = None
            confidence_tier = "none"
            flat_reason = "macro_direction unavailable"

    # ── REGIME overlay ───────────────────────────────────────────────────────
    macro_z = row.get("macro_attention_zscore")
    if pd.notna(macro_z):
        regime = "high_attention" if float(macro_z) > REGIME_ATTENTION_THRESHOLD else "normal"
    else:
        regime = "unknown"

    return CoordinatorSignal(
        pair=pair, date=date,
        vol_signal=vol_signal, vol_source=vol_source,
        direction=direction, direction_source=direction_source,
        direction_horizon=direction_horizon, direction_ic=direction_ic,
        confidence_tier=confidence_tier,
        flat_reason=flat_reason, regime=regime,
    )


print("coordinator_predict() defined")

coordinator_predict() defined


## 3. Smoke Test — Single Date Predictions

In [6]:
# Sample a few dates to verify logic
sample_dates = pd.to_datetime(["2024-01-15", "2024-06-10", "2025-03-20"])

for d in sample_dates:
    print(f"\n=== {d.date()} ===")
    subset = df[df["date"] == d]
    for _, row in subset.iterrows():
        sig = coordinator_predict(row)
        print(f"  {sig}")


=== 2024-01-15 ===
  CoordinatorSignal(EURUSD 2024-01-15 | vol=0.3097[geo] | DOWN[macro_eurusd,tier=low,h=5d,ic=+0.060] | regime=normal)
  CoordinatorSignal(GBPUSD 2024-01-15 | vol=0.3097[geo] | UP[macro_gbpusd,tier=medium,h=5d,ic=+0.115] | regime=normal)
  CoordinatorSignal(USDCHF 2024-01-15 | vol=0.3097[geo] | DOWN[macro_usdchf,tier=low,h=5d,ic=+0.033] | regime=normal)
  CoordinatorSignal(USDJPY 2024-01-15 | vol=0.3097[geo] | UP[macro_usdjpy,tier=low,h=5d,ic=+0.058] | regime=normal)

=== 2024-06-10 ===
  CoordinatorSignal(EURUSD 2024-06-10 | vol=0.5048[geo] | DOWN[macro_eurusd,tier=low,h=5d,ic=+0.060] | regime=normal)
  CoordinatorSignal(GBPUSD 2024-06-10 | vol=0.5048[geo] | UP[macro_gbpusd,tier=medium,h=5d,ic=+0.115] | regime=normal)
  CoordinatorSignal(USDCHF 2024-06-10 | vol=0.5048[geo] | DOWN[macro_usdchf,tier=low,h=5d,ic=+0.033] | regime=normal)
  CoordinatorSignal(USDJPY 2024-06-10 | vol=0.6385[stocktwits] | DOWN[macro_usdjpy,tier=low,h=5d,ic=+0.058] | regime=normal)

=== 2025

## 4. Batch Run — Full Signal Table

In [7]:
signals = [coordinator_predict(row) for _, row in df.iterrows()]

coord = pd.DataFrame({
    "date":               [s.date for s in signals],
    "pair":               [s.pair for s in signals],
    "vol_signal":         [s.vol_signal for s in signals],
    "vol_source":         [s.vol_source for s in signals],
    "direction":          [s.direction for s in signals],
    "direction_source":   [s.direction_source for s in signals],
    "direction_horizon":  [s.direction_horizon for s in signals],
    "direction_ic":       [s.direction_ic for s in signals],
    "confidence_tier":    [s.confidence_tier for s in signals],
    "regime":             [s.regime for s in signals],
    "flat_reason":        [s.flat_reason for s in signals],
})

coord = coord.merge(
    df[["date", "pair", "fwd_vol_3d", "fwd_ret_1d", "fwd_ret_5d", "fwd_vol_10d"]],
    on=["date", "pair"], how="left"
)

print(f"Coordinator signals: {len(coord):,} rows")
print("\nvol_source distribution:")
print(coord.groupby(["pair", "vol_source"]).size().unstack(fill_value=0))
print("\nconfidence_tier distribution:")
print(coord.groupby(["pair", "confidence_tier"]).size().unstack(fill_value=0))
print("\ndirection_source distribution:")
print(coord.groupby(["pair", "direction_source"]).size().unstack(fill_value=0))

Coordinator signals: 4,172 rows

vol_source distribution:
vol_source   geo  stocktwits
pair                        
EURUSD      1043           0
GBPUSD      1043           0
USDCHF      1043           0
USDJPY       625         418

confidence_tier distribution:
confidence_tier  high   low  medium
pair                               
EURUSD              0  1043       0
GBPUSD              0     0    1043
USDCHF              0  1043       0
USDJPY            174   869       0

direction_source distribution:
direction_source  macro_eurusd  macro_gbpusd  macro_usdchf  macro_usdjpy  \
pair                                                                       
EURUSD                    1043             0             0             0   
GBPUSD                       0          1043             0             0   
USDCHF                       0             0          1043             0   
USDJPY                       0             0             0           869   

direction_source  tech_usdjpy  


## 5. VOL Track Validation — IC vs `fwd_vol_3d`

Validates the fusion rule: StockTwits on active USDJPY days, geo everywhere else.
IC = Spearman rank correlation of `vol_signal` vs `fwd_vol_3d`.

In [8]:
def ic(x: pd.Series, y: pd.Series) -> tuple[float, float]:
    """Spearman IC and p-value, dropping NaNs."""
    valid = x.notna() & y.notna()
    if valid.sum() < 10:
        return float("nan"), float("nan")
    r, p = stats.spearmanr(x[valid], y[valid])
    return float(r), float(p)


print("=" * 65)
print("VOL Track IC — vol_signal vs fwd_vol_3d")
print("=" * 65)

# All pairs combined
r, p = ic(coord["vol_signal"], coord["fwd_vol_3d"])
print(f"\nALL pairs combined: IC={r:+.4f}  p={p:.4f}  n={coord['vol_signal'].notna().sum()}")

# Per pair
print("\nPer pair:")
for pair in sorted(coord["pair"].unique()):
    sub = coord[coord["pair"] == pair]
    r, p = ic(sub["vol_signal"], sub["fwd_vol_3d"])
    src_counts = sub["vol_source"].value_counts().to_dict()
    print(f"  {pair}: IC={r:+.4f}  p={p:.4f}  sources={src_counts}")

# By source (disaggregated)
print("\nBy vol_source (disaggregated):")
for src in ["stocktwits", "geo"]:
    sub = coord[coord["vol_source"] == src]
    r, p = ic(sub["vol_signal"], sub["fwd_vol_3d"])
    print(f"  {src}: IC={r:+.4f}  p={p:.4f}  n={len(sub)}")

VOL Track IC — vol_signal vs fwd_vol_3d

ALL pairs combined: IC=+0.0875  p=0.0000  n=4172

Per pair:
  EURUSD: IC=+0.1239  p=0.0001  sources={'geo': 1043}
  GBPUSD: IC=+0.1222  p=0.0001  sources={'geo': 1043}
  USDCHF: IC=+0.1436  p=0.0000  sources={'geo': 1043}
  USDJPY: IC=-0.0449  p=0.1478  sources={'geo': 625, 'stocktwits': 418}

By vol_source (disaggregated):
  stocktwits: IC=-0.2999  p=0.0000  n=418
  geo: IC=+0.1298  p=0.0000  n=3754


In [9]:
# Quarterly IC breakdown (2022–2025) — stability check
print("VOL Track — Quarterly IC (all pairs combined)")
print("-" * 55)
coord["quarter"] = coord["date"].dt.to_period("Q")
quarters = sorted(coord["quarter"].unique())
n_pos = 0
for q in quarters:
    sub = coord[coord["quarter"] == q]
    r, p = ic(sub["vol_signal"], sub["fwd_vol_3d"])
    sign = "✓" if r > 0 else "✗"
    n_pos += int(r > 0)
    print(f"  {q}: IC={r:+.4f}  p={p:.3f}  {sign}")
print(f"\nPositive quarters: {n_pos}/{len(quarters)}")

VOL Track — Quarterly IC (all pairs combined)
-------------------------------------------------------
  2022Q1: IC=+0.0590  p=0.347  ✓
  2022Q2: IC=+0.1859  p=0.003  ✓
  2022Q3: IC=+0.1412  p=0.022  ✓
  2022Q4: IC=+0.1426  p=0.021  ✓
  2023Q1: IC=+0.1284  p=0.039  ✓
  2023Q2: IC=+0.1672  p=0.007  ✓
  2023Q3: IC=+0.1847  p=0.003  ✓
  2023Q4: IC=+0.1150  p=0.064  ✓
  2024Q1: IC=+0.3046  p=0.000  ✓
  2024Q2: IC=+0.1541  p=0.013  ✓
  2024Q3: IC=-0.0814  p=0.187  ✗
  2024Q4: IC=-0.0186  p=0.764  ✗
  2025Q1: IC=+0.1266  p=0.043  ✓
  2025Q2: IC=-0.1618  p=0.010  ✗
  2025Q3: IC=+0.0814  p=0.187  ✓
  2025Q4: IC=+0.1730  p=0.006  ✓

Positive quarters: 13/16


## 6. Directional Track Validation — USDJPY

Only USDJPY gets a directional call (when tech_confidence ≥ pair-p75).
Validation metric: binary accuracy (direction=1 when next-day return > 0).

In [10]:
# USDJPY rows where directional gate fired
gated = coord[(coord["pair"] == "USDJPY") & (coord["direction_source"] == "tech_usdjpy")].copy()
gated = gated.dropna(subset=["fwd_ret_1d"])

print(f"USDJPY directional-gated days: {len(gated)} ({len(gated) / coord['pair'].value_counts()['USDJPY']:.1%} of USDJPY)")
print(f"  date range: {gated['date'].min().date()} → {gated['date'].max().date()}")

# Convert to binary: fwd_ret_1d > 0 → actual_up = 1
gated["actual_up"] = (gated["fwd_ret_1d"] > 0).astype(int)

acc = (gated["direction"] == gated["actual_up"]).mean()
n = len(gated)

# Binomial significance test
from scipy.stats import binomtest
result = binomtest(int((gated["direction"] == gated["actual_up"]).sum()), n=n, p=0.5, alternative="greater")

print(f"\nDirectional accuracy (gated days): {acc:.3f}")
print(f"Binomial p-value (H0: acc=0.5): {result.pvalue:.4f}")

# Breakdown by direction call
print("\nBy direction call:")
for d, label in [(1, "UP calls"), (0, "DOWN calls")]:
    sub = gated[gated["direction"] == d]
    if len(sub) == 0:
        print(f"  {label}: n=0")
        continue
    a = (sub["direction"] == sub["actual_up"]).mean()
    print(f"  {label}: n={len(sub)}, accuracy={a:.3f}")

USDJPY directional-gated days: 174 (16.7% of USDJPY)
  date range: 2022-03-18 → 2025-11-21

Directional accuracy (gated days): 0.586
Binomial p-value (H0: acc=0.5): 0.0138

By direction call:
  UP calls: n=174, accuracy=0.586
  DOWN calls: n=0


In [11]:
# Quarterly directional accuracy (USDJPY gated)
print("USDJPY Directional — Quarterly Accuracy")
print("-" * 45)
gated["quarter"] = gated["date"].dt.to_period("Q")
q_stats = gated.groupby("quarter").apply(
    lambda g: pd.Series({
        "n": len(g),
        "acc": (g["direction"] == g["actual_up"]).mean()
    })
).reset_index()

for _, row in q_stats.iterrows():
    sign = "✓" if row["acc"] > 0.5 else "✗"
    print(f"  {row['quarter']}: n={int(row['n']):3d}  acc={row['acc']:.3f}  {sign}")

n_above = (q_stats["acc"] > 0.5).sum()
print(f"\nQuarters above 50%: {n_above}/{len(q_stats)}")

USDJPY Directional — Quarterly Accuracy
---------------------------------------------
  2022Q1: n= 10  acc=0.500  ✗
  2022Q2: n= 47  acc=0.617  ✓
  2022Q3: n= 26  acc=0.615  ✓
  2022Q4: n= 10  acc=0.700  ✓
  2023Q1: n= 14  acc=0.429  ✗
  2023Q2: n= 13  acc=0.615  ✓
  2023Q3: n= 10  acc=0.300  ✗
  2024Q1: n=  3  acc=0.667  ✓
  2024Q2: n=  7  acc=0.571  ✓
  2024Q3: n=  3  acc=0.333  ✗
  2024Q4: n= 30  acc=0.667  ✓
  2025Q4: n=  1  acc=1.000  ✓

Quarters above 50%: 8/12


C:\Users\yassi\AppData\Local\Temp\ipykernel_41888\3870486092.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  q_stats = gated.groupby("quarter").apply(


## 7. Regime Overlay Validation

`macro_attention_zscore > 1.0` → `high_attention` regime. Tier B signal — in-sample only.
Check whether vol is empirically higher during `high_attention` periods.

In [12]:
# Does high_attention regime predict higher realized vol?
print("Regime overlay — realized fwd_vol_3d by regime (all pairs)")
print("-" * 55)
regime_vol = coord.dropna(subset=["fwd_vol_3d"]).groupby("regime")["fwd_vol_3d"].agg(["mean", "median", "count"])
print(regime_vol.round(6))

# Mann-Whitney U test: is high_attention vol > normal vol?
high = coord[coord["regime"] == "high_attention"]["fwd_vol_3d"].dropna()
normal = coord[coord["regime"] == "normal"]["fwd_vol_3d"].dropna()
if len(high) > 10 and len(normal) > 10:
    u_stat, p_mw = stats.mannwhitneyu(high, normal, alternative="greater")
    print(f"\nMann-Whitney U: high>normal vol? p={p_mw:.4f}")

# IC of macro_attention_zscore vs fwd_vol_3d
macro_z = df.set_index(["date", "pair"])["macro_attention_zscore"]
fwd_v3 = df.set_index(["date", "pair"])["fwd_vol_3d"]
r_macro, p_macro = ic(macro_z, fwd_v3)
print(f"\nIC(macro_attention_zscore → fwd_vol_3d): {r_macro:+.4f}  p={p_macro:.4f}")

Regime overlay — realized fwd_vol_3d by regime (all pairs)
-------------------------------------------------------
                    mean    median  count
regime                                   
high_attention  0.004311  0.003551   1460
normal          0.004049  0.003385   2696

Mann-Whitney U: high>normal vol? p=0.0397

IC(macro_attention_zscore → fwd_vol_3d): +0.0870  p=0.0000


## 8. Full-Period Summary

In [13]:
print("=" * 70)
print("COORDINATOR NB03 — SUMMARY")
print("=" * 70)

# VOL track
r_all, p_all = ic(coord["vol_signal"], coord["fwd_vol_3d"])
r_st,  p_st  = ic(coord[coord["vol_source"]=="stocktwits"]["vol_signal"],
                  coord[coord["vol_source"]=="stocktwits"]["fwd_vol_3d"])
r_geo, p_geo = ic(coord[coord["vol_source"]=="geo"]["vol_signal"],
                  coord[coord["vol_source"]=="geo"]["fwd_vol_3d"])
st_n  = (coord["vol_source"]=="stocktwits").sum()
geo_n = (coord["vol_source"]=="geo").sum()

print("\nVOL track (fwd_vol_3d):")
print(f"  Fused (all pairs):          IC={r_all:+.4f}  p={p_all:.4f}  n={len(coord)}")
print(f"  StockTwits source ({st_n:4d}d):  IC={r_st:+.4f}  p={p_st:.4f}  [sign-inverted]")
print(f"  Geo source        ({geo_n:4d}d):  IC={r_geo:+.4f}  p={p_geo:.4f}")

# Directional track
print("\nDIRECTIONAL track:")
usdjpy_tech = coord[(coord["pair"]=="USDJPY") & (coord["direction_source"]=="tech_usdjpy")].dropna(subset=["fwd_ret_1d"])
if len(usdjpy_tech) > 0:
    acc = (usdjpy_tech["direction"] == (usdjpy_tech["fwd_ret_1d"] > 0).astype(int)).mean()
    pct = len(usdjpy_tech) / (coord["pair"]=="USDJPY").sum()
    print(f"  USDJPY  tech_usdjpy (n={len(usdjpy_tech):4d}, {pct:.1%}): accuracy={acc:.3f}  tier=high  h=1d")

for pair, (tier, horizon, ic_val) in PAIR_MACRO_CONFIG.items():
    src = "tech_usdjpy" if pair == "USDJPY" else f"macro_{pair.lower()}"
    macro_rows = coord[(coord["pair"]==pair) & (coord["direction_source"].str.startswith("macro"))].dropna(subset=["fwd_ret_5d", "direction"])
    if len(macro_rows) > 0:
        r5, p5 = ic(macro_rows["direction"].astype(float), macro_rows["fwd_ret_5d"])
        pct = len(macro_rows) / (coord["pair"]==pair).sum()
        label = "macro_usdjpy" if pair == "USDJPY" else f"macro_{pair.lower()}"
        print(f"  {pair}  {label:<16} (n={len(macro_rows):4d}, {pct:.1%}): IC={r5:+.4f}  p={p5:.3f}  tier={tier}  h=5d")

# Direction coverage
print("\nDirection coverage (all pairs):")
for pair in sorted(coord["pair"].unique()):
    pct = coord[coord["pair"]==pair]["direction"].notna().mean()
    print(f"  {pair}: {pct:.1%} have a call")

# Regime overlay
print(f"\nREGIME overlay:")
print(f"  IC(macro_attention → fwd_vol_3d): {r_macro:+.4f}  p={p_macro:.4f}  (Tier B in-sample)")
pct_high = (coord["regime"]=="high_attention").mean()
print(f"  high_attention on {pct_high:.1%} of all rows")

print("\nConstraints:")
print("  - geo zone collapse: identical geo vol signal across all 4 pairs")
print("  - USDJPY tech: constant direction=1 → accuracy=58.6%, not IC")
print("  - macro validated at 5d; tech validated at 1d — not directly comparable")
print("  - regime overlay: in-sample only")
print("=" * 70)

COORDINATOR NB03 — SUMMARY

VOL track (fwd_vol_3d):
  Fused (all pairs):          IC=+0.0875  p=0.0000  n=4172
  StockTwits source ( 418d):  IC=-0.2999  p=0.0000  [sign-inverted]
  Geo source        (3754d):  IC=+0.1298  p=0.0000

DIRECTIONAL track:
  USDJPY  tech_usdjpy (n= 174, 16.7%): accuracy=0.586  tier=high  h=1d
  GBPUSD  macro_gbpusd     (n=1037, 99.4%): IC=+0.1147  p=0.000  tier=medium  h=5d
  EURUSD  macro_eurusd     (n=1037, 99.4%): IC=+0.0595  p=0.055  tier=low  h=5d
  USDCHF  macro_usdchf     (n=1037, 99.4%): IC=+0.0325  p=0.296  tier=low  h=5d
  USDJPY  macro_usdjpy     (n= 863, 82.7%): IC=+0.0536  p=0.116  tier=low  h=5d

Direction coverage (all pairs):
  EURUSD: 100.0% have a call
  GBPUSD: 100.0% have a call
  USDCHF: 100.0% have a call
  USDJPY: 100.0% have a call

REGIME overlay:
  IC(macro_attention → fwd_vol_3d): +0.0870  p=0.0000  (Tier B in-sample)
  high_attention on 35.3% of all rows

Constraints:
  - geo zone collapse: identical geo vol signal across all 4 pai

## 9. Sign Convention — StockTwits vs Geo

`usdjpy_stocktwits_vol_signal` has IC = **−0.30** vs fwd_vol_3d (NB02 Part 2 confirmed).

This means: **higher stocktwits value → lower realized vol** (inverse relationship). The signal is still Tier A — the absolute IC magnitude is what counts, not the sign.

When `vol_source == "stocktwits"`, multiply `vol_signal` by **−1** to get a consistently-signed vol forecast where higher = more vol expected. Geo signal is correctly signed (higher `geo_bilateral_risk` → higher realized vol).

**Rule for downstream consumers:** do not compare `vol_signal` values across sources directly without sign normalization. Always check `vol_source`.


In [14]:
# ── Pre-implementation validation: per-pair macro_direction IC ───────────────
# macro_direction is "up"/"down" — encode as 1/0 to compute IC
df["macro_dir_enc"] = (df["macro_direction"] == "up").astype(float)

print("Per-pair macro_direction IC vs fwd_ret_1d and fwd_ret_5d")
print("=" * 60)
THRESHOLD = 0.02  # minimum IC to use macro as a low-conf directional signal
print(f"Threshold for 'low' tier: IC >= {THRESHOLD} at either horizon\n")

tier_decisions = {}
for pair in sorted(df["pair"].unique()):
    sub = df[df["pair"] == pair]
    r1, p1 = ic(sub["macro_dir_enc"], sub["fwd_ret_1d"])
    r5, p5 = ic(sub["macro_dir_enc"], sub["fwd_ret_5d"])
    passes = (r1 >= THRESHOLD) or (r5 >= THRESHOLD)
    tier = "low" if passes else "none (FLAT)"
    tier_decisions[pair] = tier
    print(f"  {pair}: IC_1d={r1:+.4f}(p={p1:.3f})  IC_5d={r5:+.4f}(p={p5:.3f})  → {tier}")

print("\nDecisions:")
for pair, tier in tier_decisions.items():
    print(f"  {pair}: confidence_tier = '{tier}'")

Per-pair macro_direction IC vs fwd_ret_1d and fwd_ret_5d
Threshold for 'low' tier: IC >= 0.02 at either horizon

  EURUSD: IC_1d=-0.0103(p=0.739)  IC_5d=+0.0595(p=0.055)  → low
  GBPUSD: IC_1d=+0.0917(p=0.003)  IC_5d=+0.1147(p=0.000)  → low
  USDCHF: IC_1d=+0.0234(p=0.451)  IC_5d=+0.0325(p=0.296)  → low
  USDJPY: IC_1d=+0.0034(p=0.913)  IC_5d=+0.0578(p=0.063)  → low

Decisions:
  EURUSD: confidence_tier = 'low'
  GBPUSD: confidence_tier = 'low'
  USDCHF: confidence_tier = 'low'
  USDJPY: confidence_tier = 'low'


## 10. Multi-Pair Directional Validation

Per-pair accuracy of the coordinator's directional call at the validated horizon:
- `tech_usdjpy` rows → accuracy vs fwd_ret_1d > 0
- `macro_*` rows → IC vs fwd_ret_5d (Spearman, consistent with training-side validation)

In [15]:
print("Multi-pair directional validation")
print("=" * 65)

for pair in sorted(coord["pair"].unique()):
    sub = coord[coord["pair"] == pair].copy()
    print(f"\n{pair}")

    # tech_usdjpy rows → accuracy at 1d
    tech_rows = sub[sub["direction_source"] == "tech_usdjpy"].dropna(subset=["fwd_ret_1d"])
    if len(tech_rows) > 0:
        actual_up = (tech_rows["fwd_ret_1d"] > 0).astype(int)
        acc = (tech_rows["direction"] == actual_up).mean()
        n = len(tech_rows)
        print(f"  tech_usdjpy  (n={n:4d}, h=1d): accuracy={acc:.3f}  tier=high")

    # macro rows → IC at 5d
    macro_rows = sub[sub["direction_source"].str.startswith("macro")].dropna(subset=["fwd_ret_5d", "direction"])
    if len(macro_rows) > 0:
        r5, p5 = ic(macro_rows["direction"].astype(float), macro_rows["fwd_ret_5d"])
        tier = macro_rows["confidence_tier"].iloc[0]
        src = macro_rows["direction_source"].iloc[0]
        n = len(macro_rows)
        print(f"  {src:<16} (n={n:4d}, h=5d): IC={r5:+.4f}  p={p5:.3f}  tier={tier}")

print("\n\nDirection coverage (% of rows with a call):")
for pair in sorted(coord["pair"].unique()):
    sub = coord[coord["pair"] == pair]
    pct = sub["direction"].notna().mean()
    print(f"  {pair}: {pct:.1%}")

Multi-pair directional validation

EURUSD
  macro_eurusd     (n=1037, h=5d): IC=+0.0595  p=0.055  tier=low

GBPUSD
  macro_gbpusd     (n=1037, h=5d): IC=+0.1147  p=0.000  tier=medium

USDCHF
  macro_usdchf     (n=1037, h=5d): IC=+0.0325  p=0.296  tier=low

USDJPY
  tech_usdjpy  (n= 174, h=1d): accuracy=0.586  tier=high
  macro_usdjpy     (n= 863, h=5d): IC=+0.0536  p=0.116  tier=low


Direction coverage (% of rows with a call):
  EURUSD: 100.0%
  GBPUSD: 100.0%
  USDCHF: 100.0%
  USDJPY: 100.0%
